# Energia gazdálkodás optimalizálása Spark használatával

MySql adatbázisban gyűjtünk adatokat rendszerünk fogyasztásáról, a rendelkezésre álló napelemek termeléséről, illetve az energia ár alakulásáról 15 perces felbontásban.

Az árakat a hupx oldalról API-n keresztül szereztük.

A fogyasztás, termelés és ár adatok rendre a **consumer_profile**, **solar_panel_production** és **hupx_energy_price** táblákban kerülnek letárolásra időbélyeggel együt.



In [2]:
import findspark
import pyspark.sql
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp, col, to_utc_timestamp, count, monotonically_increasing_id


findspark.init()

spark = SparkSession.builder \
    .appName("PySpark MySQL Connection") \
    .getOrCreate()


24/05/18 14:45:01 WARN Utils: Your hostname, PySpark resolves to a loopback address: 127.0.1.1; using 192.168.64.129 instead (on interface ens33)
24/05/18 14:45:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/student/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/student/.ivy2/cache
The jars for the packages stored in: /home/student/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d7943cf2-e764-409b-9a78-d21c79746c00;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.3.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.3.1 in central
	found org.apache.kafka#kafka-clients;2.8.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.8.4 in central
	found org.slf4j#slf4j-api;1.7.32 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.2 in central
	found org.spark-project.spark#unused;1.0.0 in central
	found org.apache.hadoop#hadoop-client-api;3.3.2 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: re

# Manuális ETL
Ez a blokk tesztadatok manuális betöltését szolgálja fáljokból, valamint egyéb kiegészítő adatok szolgáltatását, ha szükséges.

Itt kell elvégezni az adatok transzformálását, (pl: megfelelő típusra való cast-olás).

A következő blokkok futtatása kihagyható, ha nincsenek ilyen igények.

# Kiolvasás

In [28]:

#reading and transforming csv input, only run this block when needed.
consumption = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",";")\
          .option("inferSchema","true")\
          .load("../ingest/consumption_profile.csv")

production = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",",")\
          .option("inferSchema","true")\
          .load("../ingest/production_profile.csv")

price = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",";")\
          .option("inferSchema","true")\
          .load("../ingest/price.csv")

consumption.show()
production.show(100)
price.show()






+----------+-------------------+--------+
|     Dátum|         Negyedórák| Group#4|
+----------+-------------------+--------+
|2022.01.01|2024-05-18 00:00:00|0.018103|
|2022.01.01|2024-05-18 00:15:00|0.018038|
|2022.01.01|2024-05-18 00:30:00| 0.01801|
|2022.01.01|2024-05-18 00:45:00|0.018095|
|2022.01.01|2024-05-18 01:00:00|0.018187|
|2022.01.01|2024-05-18 01:15:00|0.018158|
|2022.01.01|2024-05-18 01:30:00|0.018087|
|2022.01.01|2024-05-18 01:45:00|0.018103|
|2022.01.01|2024-05-18 02:00:00|0.018172|
|2022.01.01|2024-05-18 02:15:00|0.018167|
|2022.01.01|2024-05-18 02:30:00|0.018129|
|2022.01.01|2024-05-18 02:45:00|0.018056|
|2022.01.01|2024-05-18 03:00:00|0.017923|
|2022.01.01|2024-05-18 03:15:00|0.017764|
|2022.01.01|2024-05-18 03:30:00|0.017725|
|2022.01.01|2024-05-18 03:45:00| 0.01777|
|2022.01.01|2024-05-18 04:00:00|0.017912|
|2022.01.01|2024-05-18 04:15:00|0.018011|
|2022.01.01|2024-05-18 04:30:00| 0.01817|
|2022.01.01|2024-05-18 04:45:00| 0.01846|
+----------+-------------------+--

In [29]:
production = production.withColumn("time",to_timestamp(col("time"),"dd.MM.yyyy HH:mm"))\
                       .withColumn("utc_time",to_utc_timestamp(col("time"),"Europe/Budapest"))\
                       .withColumn("rownum", monotonically_increasing_id())




DataFrame[time: timestamp, production_(W): double, utc_time: timestamp, rownum: bigint]

In [36]:
production.filter(col("time") >= "2022-10-30").show(50)

+-------------------+--------------+-------------------+------+
|               time|production_(W)|           utc_time|rownum|
+-------------------+--------------+-------------------+------+
|2022-10-30 00:00:00|           0.0|2022-10-29 22:00:00| 28992|
|2022-10-30 00:15:00|           0.0|2022-10-29 22:15:00| 28993|
|2022-10-30 00:30:00|           0.0|2022-10-29 22:30:00| 28994|
|2022-10-30 00:45:00|           0.0|2022-10-29 22:45:00| 28995|
|2022-10-30 01:00:00|           0.0|2022-10-29 23:00:00| 28996|
|2022-10-30 01:15:00|           0.0|2022-10-29 23:15:00| 28997|
|2022-10-30 01:30:00|           0.0|2022-10-29 23:30:00| 28998|
|2022-10-30 01:45:00|           0.0|2022-10-29 23:45:00| 28999|
|2022-10-30 02:00:00|           0.0|2022-10-30 00:00:00| 29000|
|2022-10-30 02:15:00|           0.0|2022-10-30 00:15:00| 29001|
|2022-10-30 02:30:00|           0.0|2022-10-30 00:30:00| 29002|
|2022-10-30 02:45:00|           0.0|2022-10-30 00:45:00| 29003|
|2022-10-30 03:00:00|           0.0|2022

In [34]:
filter = production.groupBy("utc_time").agg(count("time").alias("count")).filter(col("count")> 1)

filtered = production.join(filter,"utc_time").select("time","utc_time","rownum")
filtered.show()

+-------------------+-------------------+------+
|               time|           utc_time|rownum|
+-------------------+-------------------+------+
|2022-03-27 03:00:00|2022-03-27 01:00:00|  8168|
|2022-03-27 03:15:00|2022-03-27 01:15:00|  8169|
|2022-03-27 03:30:00|2022-03-27 01:30:00|  8170|
|2022-03-27 03:45:00|2022-03-27 01:45:00|  8171|
|2022-03-27 03:00:00|2022-03-27 01:00:00|  8172|
|2022-03-27 03:15:00|2022-03-27 01:15:00|  8173|
|2022-03-27 03:30:00|2022-03-27 01:30:00|  8174|
|2022-03-27 03:45:00|2022-03-27 01:45:00|  8175|
|2022-03-27 04:00:00|2022-03-27 03:00:00|  8176|
|2022-03-27 04:15:00|2022-03-27 03:15:00|  8177|
|2022-03-27 04:30:00|2022-03-27 03:30:00|  8178|
|2022-03-27 04:45:00|2022-03-27 03:45:00|  8179|
|2022-03-27 05:00:00|2022-03-27 03:00:00|  8180|
|2022-03-27 05:15:00|2022-03-27 03:15:00|  8181|
|2022-03-27 05:30:00|2022-03-27 03:30:00|  8182|
|2022-03-27 05:45:00|2022-03-27 03:45:00|  8183|
|2023-03-26 03:00:00|2023-03-26 01:00:00| 43112|
|2023-03-26 03:15:00

# Transzformálás

In [ ]:
df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "dd.MM.yyyy HH:mm"))
#df.printSchema()
#df.show()

# Betöltés

In [ ]:
#inserting data into DB
df.write\
  .format("jdbc")\
  .mode("append")\
  .option("driver","com.mysql.jdbc.Driver")\
  .option("url", "jdbc:mysql://localhost:3306/energycom")\
  .option("dbtable", "consumer_profile")\
  .option("user", "root")\
  .option("password", "mysql")\
  .save()

# Adatok beolvasása az adatbázisból

In [2]:
# Adatok betöltése a MySQL adatbázisból
consumer = spark.read \
    .format("jdbc") \
    .option("url", mysql_url) \
    .option("driver", "com.mysql.cj.jdbc.Driver")\
    .option("dbtable", "consumer_profile") \
    .load()

production = spark.read \
    .format("jdbc") \
    .option("url", mysql_url) \
    .option("driver", "com.mysql.cj.jdbc.Driver")\
    .option("dbtable", "solar_panel_production") \
    .load()

price = spark.read \
    .format("jdbc") \
    .option("url", mysql_url) \
    .option("driver", "com.mysql.cj.jdbc.Driver")\
    .option("dbtable", "hupx_energy_price") \
    .load()

consumer.show(10)
production.show(10)
price.show(10)

+----------+-------------------+---------------+
|profile_id|          timestamp|consumption_kwh|
+----------+-------------------+---------------+
|         1|2023-01-01 00:00:00|        0.02656|
|         1|2023-01-01 00:15:00|        0.02601|
|         1|2023-01-01 00:30:00|        0.02552|
|         1|2023-01-01 00:45:00|        0.02483|
|         1|2023-01-01 01:00:00|        0.02421|
|         1|2023-01-01 01:15:00|        0.02367|
|         1|2023-01-01 01:30:00|        0.02321|
|         1|2023-01-01 01:45:00|        0.02281|
|         1|2023-01-01 02:00:00|        0.02246|
|         1|2023-01-01 02:15:00|        0.02214|
+----------+-------------------+---------------+
only showing top 10 rows

+---------+-------------------+--------------+
|device_id|          timestamp|production_kwh|
+---------+-------------------+--------------+
|        1|2023-01-01 00:00:00|       0.00000|
|        1|2023-01-01 00:15:00|       0.00000|
|        1|2023-01-01 00:30:00|       0.00000|
|     

# Kiindulási adatok egyesítése

Az optimalizáció előkészítéséhez egyesítjük az adatokat egyetlen Pandas DataFrame-ben.

Előzetes szűrés szükséges a fogyasztó rendszer és a termelő egységre, az adatbázisban több egység termelési illetve fogyasztási adatait is tárolhatjuk. A szűrést azonosítók segítségével tehetjük meg.
Egységesség érdekében float típusú mezővé alakítom a három mutatót a számítási feladatok érdekében.

Itt kerülnek létrehozásra az általánosan szükséges paraméterek is, mint például a rendelkezésre álló akkumulátor kapacitása is.

In [4]:
from pyspark.sql.functions import *
# paramaméterek
capacity = 10 #kwh

# szűrés
consumer   = consumer.filter(col("profile_id") == 1)
production = production.filter(col("device_id") == 1)

# összekapcsolás
process = consumer.select("timestamp","consumption_kwh").join(production.select("timestamp","production_kwh"),"timestamp","inner")\
                  .join(price.select("timestamp","price_eur_per_mwh"),"timestamp","inner")

#rendezés join után és cast-olás
process = process.orderBy("timestamp")\
                 .withColumn("consumption_kwh", process["consumption_kwh"].cast("float"))\
                 .withColumn("production_kwh", process["production_kwh"].cast("float"))\
                 .withColumn("price_eur_per_mwh", process["price_eur_per_mwh"].cast("float"))\
                 .toPandas()
process.head()


,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh
0,2023-01-01 00:00:00,0.02656,0.0,19.76
1,2023-01-01 00:15:00,0.02601,0.0,19.76
2,2023-01-01 00:30:00,0.02552,0.0,19.76
3,2023-01-01 00:45:00,0.02483,0.0,19.76
4,2023-01-01 01:00:00,0.02421,0.0,0.19


# Optimalizálási feladat

Most, hogy kész a rendezett adatsorunk, alkalmazhatjuk a különböző optimalizációs stratégiákat a minél eredményesebb energia gazdálkodás érdekében. 

A célfüggvény amit választottunk az a fogyasztási igények kiszolgálása a **költség minimalizálásával**. Annál jobb az eredmény, minél kevesebbe került az energiaszolgáltatás az adott időszakra.

különböző megvalósításokat igyekszünk összehasonlítani, hogy megállapíthassuk, melyik megoldás a legkedvezőbb.

# Egyszerű stratégia

A következő blokkokban egy próba algoritmust implementáltam, mely nem az adatok viszonya alapján készíti el a költség tervet.

Csupán referenciaként szolgáló megoldás.

**Működés:**

Ha az adott időben a termelés fedezi a fogyasztást, akkor többlet keletkezik, a többletből betáplálunk annyit az akkumulátorba, amennyi belefér, az el nem tárolható mennyiséget pedig kitápláljuk a hálózatra.

Másik esetben, mikor nem fedezi a termelés a fogyasztást, akkor a szükséges energiamennyiséget első sorban az akkumulátorban tárolt energiából pótoljuk, hogyha az sem elég, akkor vételezünk a hálózatról.


In [6]:
import numpy as np
# energia többlet és igény kiszámítása a termelés és fogyasztás különbségéből,
# valamint töltöttséget követő oszlopok bevezetése a negyedóra elejére és végére, alapértelmezett értékadás.
basic_p = process.assign(energy_to_store = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, x['production_kwh'] - x['consumption_kwh'], 0),
                       energy_to_get   = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, x['consumption_kwh'] - x['production_kwh'], 0),
                       battery_charge_start = float(0.00000),
                       battery_charge_end   = float(0.00000))
basic_p.head()

#from pyspark.sql.functions import when
#basic_p = process.withColumn("energy_to_store", when(col("production_kwh")-col("consumption_kwh")>0,col("production_kwh")-col("consumption_kwh"))\
#                          .otherwise(0))\
#               .withColumn("energy_to_get",when(col("consumption_kwh")-col("production_kwh")>0,col("consumption_kwh")-col("production_kwh"))\
#                          .otherwise(0))\
#               .withColumn("battery_charge_at_start",lit(0.00000))\
#               .withColumn("battery_charge_at_end",lit(0.00000))

# kezdő értéket állíthatunk az akkumulátorunknak a vizsgált időszak elején.
# basic_p.at[0,"battery_charge_start"] = 2

#ha tárolni kell akkor mennyit tudunk eltárolni
if basic_p.at[0,'energy_to_store'] > 0:
        temp = basic_p.at[0,'battery_charge_start'] + basic_p.at[0,'energy_to_store']
        if temp <= capacity:
            basic_p.at[0,'battery_charge_end'] = temp
        else:
            basic_p.at[0,'battery_charge_end'] = capacity
            
#ha be kell szerezni, mennyit tudunk szolgáltatni saját tárunkból
else:
        temp = basic_p.at[0,'battery_charge_start'] - basic_p.at[0,'energy_to_get']
        if temp > 0:
            basic_p.at[0,'battery_charge_end'] = temp
        else:
            basic_p.at[0,'battery_charge_end'] = 0

for i in range(1,int(basic_p.size/8)):
    basic_p.at[i,'battery_charge_start'] = basic_p.at[i-1,'battery_charge_end']
    if basic_p.at[i,'energy_to_store'] > 0:
        temp = basic_p.at[i,'battery_charge_start'] + basic_p.at[i,'energy_to_store']
        if temp <= capacity:
            basic_p.at[i,'battery_charge_end'] = temp
        else:
            basic_p.at[i,'battery_charge_end'] = capacity
    else:
        temp = basic_p.at[i,'battery_charge_start'] - basic_p.at[i,'energy_to_get']
        if temp > 0:
            basic_p.at[i,'battery_charge_end'] = temp
        else:
            basic_p.at[i,'battery_charge_end'] = 0
#basic_p.tail(10)

#az előző adatok alapján megállapítani, hogy mennyit fogyasztunk saját, illetve hálózati energiából, 
# valamint mennyi energiát táplálunk be saját akkumulátorunkba, vagy a hálózatba. 
basic_p = basic_p.assign(taking_from_battery = lambda x: np.where(x['battery_charge_start'] - x['battery_charge_end'] > 0, x['battery_charge_start'] - x['battery_charge_end'], 0),
                       feeding_battery   = lambda x: np.where(x['battery_charge_end'] - x['battery_charge_start'] > 0, x['battery_charge_end'] - x['battery_charge_start'], 0),
                       taking_from_grid  = lambda x: x['energy_to_get']-x['taking_from_battery'],
                       feeding_grid      = lambda x: x['energy_to_store']-x['feeding_battery'],
                       price             = lambda x: (x['taking_from_grid']*x['price_eur_per_mwh']-x['feeding_grid']*x['price_eur_per_mwh'])*0.001) # mwh-->kwh konverzió

basic_p.tail()


,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh,energy_to_store,energy_to_get,battery_charge_start,battery_charge_end,taking_from_battery,feeding_battery,taking_from_grid,feeding_grid,price
2971,2023-01-31 22:45:00,0.03004,0.0,121.650002,0.0,0.03004,8.86815,8.83811,0.03004,0.0,0.0,0.0,0.0
2972,2023-01-31 23:00:00,0.02862,0.0,98.690002,0.0,0.02862,8.83811,8.80949,0.02862,0.0,0.0,0.0,0.0
2973,2023-01-31 23:15:00,0.02734,0.0,98.690002,0.0,0.02734,8.80949,8.78215,0.02734,0.0,0.0,0.0,0.0
2974,2023-01-31 23:30:00,0.02614,0.0,98.690002,0.0,0.02614,8.78215,8.75601,0.02614,0.0,0.0,0.0,0.0
2975,2023-01-31 23:45:00,0.02507,0.0,98.690002,0.0,0.02507,8.75601,8.73094,0.02507,0.0,0.0,0.0,0.0
